# Lip-sync hybrid render — Ditto + LatentSync-256 (Colab T4)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vuquoctrungbk/lipsync/blob/main/tools/colab/lipsync_render.ipynb)

Offload tầng render nặng của app [lipsync](https://github.com/vuquoctrungbk/lipsync) lên Colab:
**Ditto** (motion tự nhiên) + **LatentSync-256** (miệng khớp audio). Matte + nền xanh vẫn chạy local.

## Cách dùng (3 bước)

1. **Runtime → Change runtime type → T4 GPU** (bản free đủ cho config 256).
2. Tạo job trên Google Drive: thư mục `My Drive/lipsync-jobs/in/<tên-job>/` gồm:
   - `image.png` (hoặc .jpg/.webp) — ảnh chân dung MC
   - `audio.wav` — giọng nói (TTS hoặc thu thật)
   - `params.json` *(tùy chọn)* — `{"inference_steps": 20, "guidance_scale": 1.5, "max_width": 960}`
     (`max_width`: video Ditto được thu nhỏ trước khi vào LatentSync — RAM VM free không kham nổi
     ghi file ở 1402px; đặt `0` để giữ nguyên độ phân giải nếu chạy máy nhiều RAM)
3. **Runtime → Run all**. Kết quả về `lipsync-jobs/out/<tên-job>/`:
   `hybrid_256.mp4` (thành phẩm) + `ditto_raw.mp4` (motion thô) + `timings.json` (số đo).

Sau đó về máy local — matte nền xanh + đo sync:

```
.venv\Scripts\python.exe scripts\matte_video.py --video outputs\colab\hybrid_256.mp4
.venv\Scripts\python.exe scripts\sync_metrics.py --video outputs\lipsync_green_<runid>.mp4
```

**Thiết kế:** 2 venv riêng vì hai stack không chung sống được trong 1 môi trường
(Ditto torch 2.3.1+cu121 vs LatentSync torch 2.5.1 — mỗi bên một bộ pin; mirror đúng
kiến trúc 2-venv đã chạy thành công trên máy local, Python 3.11). Checkpoints tải từ HF
mỗi phiên (~7 GB, 5–10 phút), job bàn giao qua Drive. Setup lần đầu **~10–15 phút**,
các job sau trong phiên chạy ngay. Session Colab free giới hạn ≤12h, idle ~90 phút —
gom job chạy theo phiên.


In [ ]:
# Buoc 0: kiem tra GPU (Runtime -> Change runtime type -> T4 GPU)
import subprocess

try:
    gpu = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True,
    ).stdout.strip()
except FileNotFoundError:  # runtime CPU khong co nvidia-smi
    gpu = ""
assert gpu, "Khong thay GPU — chon Runtime -> Change runtime type -> T4 GPU roi chay lai."
print("GPU:", gpu)
# T4 16GB du cho LatentSync-256 (~6.5-8GB du kien). Config 512 can ~18GB -> Colab Pro (L4/A100).


In [ ]:
# Buoc 1: mount Google Drive + tao cau truc jobs
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

JOBS_DIR = Path("/content/drive/MyDrive/lipsync-jobs")  # doi 1 cho nay neu muon thu muc khac
IN_DIR, OUT_DIR = JOBS_DIR / "in", JOBS_DIR / "out"
for d in (IN_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)
pending = [p.name for p in sorted(IN_DIR.iterdir()) if p.is_dir()]
print("Job input :", IN_DIR)
print("Job output:", OUT_DIR)
print("Jobs dang cho:", pending or "(chua co — tao in/<ten-job>/image.png + audio.wav)")


In [ ]:
# Buoc 2: cong cu + clone 2 repo, pin dung commit da verify tren may local (spike 2026-07)
import subprocess
import time
from pathlib import Path

DITTO_COMMIT = "c3e47ee"       # antgroup/ditto-talkinghead
LATENTSYNC_COMMIT = "a229c39"  # bytedance/LatentSync — code dung chung 1.5 (256) va 1.6 (512)

SRC_DITTO = Path("/content/ditto-src")
SRC_LS = Path("/content/latentsync-src")
TIMINGS = {}


def sh(*cmd):
    # Chay lenh, stream log tho ra cell (dung loc — tqdm/stacktrace nam trong do).
    cmd = [str(c) for c in cmd]
    print("+", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)


t0 = time.monotonic()
sh("pip", "install", "-q", "uv", "huggingface_hub")
if not (SRC_DITTO / ".git").exists():
    sh("git", "clone", "https://github.com/antgroup/ditto-talkinghead", SRC_DITTO)
if not (SRC_LS / ".git").exists():
    sh("git", "clone", "https://github.com/bytedance/LatentSync", SRC_LS)
# checkout chay MOI lan (idempotent) — clone-xong-checkout-truot khong de repo troi HEAD
sh("git", "-C", SRC_DITTO, "checkout", DITTO_COMMIT)
sh("git", "-C", SRC_LS, "checkout", LATENTSYNC_COMMIT)
TIMINGS["setup_clone_s"] = round(time.monotonic() - t0, 1)
print("Clone xong |", TIMINGS["setup_clone_s"], "s")


In [ ]:
# Buoc 3a: venv Ditto — Python 3.11 (dung interpreter da proven tren may local)
# + torch 2.3.1 cu121 + numpy 1.26.4. Pin chat cac goi quyet dinh ABI/behavior
# (theo pip freeze cua venv local da chay thanh cong); goi phu de uv resolver chon.
# Sentinel .install-complete: venv chi coi la xong khi cai het goi — install fail
# giua chung thi lan chay lai se cai lai tu dau, khong dung venv nua voi.
import time
from pathlib import Path

VENVS = Path("/content/venvs")
DITTO_PY = VENVS / "ditto/bin/python"
DITTO_OK = VENVS / "ditto/.install-complete"

t0 = time.monotonic()
if not DITTO_OK.exists():
    sh("uv", "venv", VENVS / "ditto", "--python", "3.11")
    sh("uv", "pip", "install", "--python", DITTO_PY,
       "--extra-index-url", "https://download.pytorch.org/whl/cu121",
       "torch==2.3.1", "torchaudio==2.3.1",
       "numpy==1.26.4", "librosa==0.11.0", "mediapipe==0.10.35",
       # 1.22 = dong CUDA-12 cuoi; 1.27 link libcudart.so.13 -> chet tren image CUDA 12.8
       "onnxruntime-gpu==1.22.0",
       "opencv-python-headless", "scikit-image", "einops", "filetype",
       "colored", "imageio-ffmpeg", "tqdm", "cython")
    DITTO_OK.touch()
TIMINGS["venv_ditto_s"] = round(time.monotonic() - t0, 1)
print("Ditto venv:", DITTO_PY, "|", TIMINGS["venv_ditto_s"], "s")


In [ ]:
# Buoc 3b: venv LatentSync — Python 3.11, cai nguyen requirements.txt cua repo
# (torch 2.5.1 stack). insightface build tu source o day (vai phut) — binh thuong.
import time

LS_PY = VENVS / "latentsync/bin/python"
LS_OK = VENVS / "latentsync/.install-complete"

t0 = time.monotonic()
if not LS_OK.exists():
    sh("uv", "venv", VENVS / "latentsync", "--python", "3.11")
    sh("uv", "pip", "install", "--python", LS_PY, "-r", SRC_LS / "requirements.txt")
    LS_OK.touch()
TIMINGS["venv_latentsync_s"] = round(time.monotonic() - t0, 1)
print("LatentSync venv:", LS_PY, "|", TIMINGS["venv_latentsync_s"], "s")


In [ ]:
# Buoc 4: checkpoints tu HuggingFace (idempotent — chay lai dung cache, khong tai lai)
import time

from huggingface_hub import hf_hub_download, snapshot_download

t0 = time.monotonic()
# Ditto PyTorch path — T4 la Turing: TRT prebuilt trong HF repo la Ampere_Plus, khong chay duoc.
snapshot_download(
    "digital-avatar/ditto-talkinghead",
    allow_patterns=["ditto_pytorch/**", "ditto_cfg/**"],
    local_dir=SRC_DITTO / "checkpoints",
)
# LatentSync-1.5 = model 256 (1.6 = 512). Code inference dung chung — chi khac ckpt + config resolution.
for fname in ("latentsync_unet.pt", "whisper/tiny.pt"):
    hf_hub_download("ByteDance/LatentSync-1.5", fname, local_dir=SRC_LS / "checkpoints")
# insightface (buffalo_l) tu tai luc LatentSync chay lan dau — Colab co mang, khong can cell rieng.
TIMINGS["checkpoints_s"] = round(time.monotonic() - t0, 1)
print("Checkpoints OK |", TIMINGS["checkpoints_s"], "s")


In [ ]:
# Buoc 5: chay moi job dang cho trong Drive/lipsync-jobs/in/
import json
import os
import shutil
import subprocess
import threading
import time
from pathlib import Path

# Kernel Jupyter leak MPLBACKEND=module://matplotlib_inline... vao child env;
# venv co lap khong co matplotlib_inline -> ep backend Agg (dinh that 2026-07-11).
SUBPROC_ENV = {**os.environ, "MPLBACKEND": "Agg"}


def run_stage(name, cmd, cwd, work):
    # Subprocess dai + tqdm day dac: log ra FILE (khong tran cell/websocket),
    # chi in tail khi loi. Tra ve so giay chay.
    log_path = work / (name + ".log")
    t0 = time.monotonic()
    with open(log_path, "w") as log:
        rc = subprocess.run([str(c) for c in cmd], cwd=cwd, stdout=log,
                            stderr=subprocess.STDOUT, env=SUBPROC_ENV).returncode
    elapsed = round(time.monotonic() - t0, 1)
    if rc != 0:
        tail = subprocess.run(["tail", "-30", str(log_path)],
                              capture_output=True, text=True).stdout
        print(f"== {name} FAILED (rc={rc}) — tail {log_path} ==")
        print(tail)
        raise RuntimeError(f"{name} failed rc={rc}")
    print(f"{name} OK — {elapsed}s")
    return elapsed


class VramPoller:
    # Poll nvidia-smi 5s/lan de bat VRAM peak trong luc render (so do cho gate T4).
    def __init__(self):
        self.peak_mb = 0
        self._stop = threading.Event()
        self._t = threading.Thread(target=self._loop, daemon=True)

    def _loop(self):
        while not self._stop.is_set():
            try:
                out = subprocess.run(
                    ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                    capture_output=True, text=True,
                ).stdout.strip()
                if out:
                    self.peak_mb = max(self.peak_mb, int(out.splitlines()[0]))
            except (ValueError, OSError):
                pass  # mot lan poll hong khong duoc giet thread do
            self._stop.wait(5)

    def __enter__(self):
        self._t.start()
        return self

    def __exit__(self, *exc):
        self._stop.set()
        self._t.join(timeout=10)


def find_image(job_dir):
    for pat in ("*.png", "*.jpg", "*.jpeg", "*.webp"):
        hits = sorted(job_dir.glob(pat))
        if hits:
            return hits[0]
    return None


def load_params(job_dir):
    pj = job_dir / "params.json"
    params = json.loads(pj.read_text()) if pj.exists() else {}
    if not isinstance(params, dict):
        raise ValueError(f"params.json phai la JSON object, nhan duoc {type(params).__name__}")
    return params


def run_job(job_dir):
    job = job_dir.name
    out_dir = OUT_DIR / job
    if (out_dir / "hybrid_256.mp4").exists():
        print(f"[{job}] da co hybrid_256.mp4 trong out/ — bo qua (xoa file do neu muon chay lai)")
        return
    image, audio = find_image(job_dir), job_dir / "audio.wav"
    if image is None or not audio.exists():
        print(f"[{job}] thieu anh (*.png/jpg/webp) hoac audio.wav — bo qua")
        return
    params = load_params(job_dir)
    steps = params.get("inference_steps", 20)
    guidance = params.get("guidance_scale", 1.5)
    max_width = params.get("max_width", 960)

    out_dir.mkdir(parents=True, exist_ok=True)
    work = Path("/content/work") / job  # render tren disk local cua Colab (nhanh hon ghi thang Drive)
    work.mkdir(parents=True, exist_ok=True)
    ditto_mp4, hybrid_mp4 = work / "ditto_raw.mp4", work / "hybrid_256.mp4"
    timings = dict(TIMINGS)
    timings["gpu"] = subprocess.run(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        capture_output=True, text=True,
    ).stdout.strip()
    timings["params"] = {"inference_steps": steps, "guidance_scale": guidance}

    with VramPoller() as vram:
        # 1) Ditto: anh + audio -> video motion (giu nguyen do phan giai anh goc)
        timings["ditto_s"] = run_stage("ditto", [
            DITTO_PY, "inference.py",
            "--data_root", "./checkpoints/ditto_pytorch",
            "--cfg_pkl", "./checkpoints/ditto_cfg/v0.4_hubert_cfg_pytorch.pkl",
            "--audio_path", audio,
            "--source_path", image,
            "--output_path", ditto_mp4], SRC_DITTO, work)

        # Downscale truoc LatentSync: buoc ghi video cuoi buffer TOAN BO frame
        # o do phan giai goc — frame 1402px lam OOM-kill VM free 12.7GB RAM
        # (gate 2026-07-11). Mat van refine o 256. params.json {"max_width": 0} de tat.
        ls_input = ditto_mp4
        if max_width:
            ls_input = work / "ditto_scaled.mp4"
            timings["downscale_s"] = run_stage("downscale", [
                "ffmpeg", "-y", "-loglevel", "error", "-i", ditto_mp4,
                "-vf", f"scale='min({max_width},iw)':-2",
                "-c:v", "libx264", "-crf", "16", "-an", ls_input], work, work)

        # 2) LatentSync-256: de mieng khop audio len video Ditto
        timings["latentsync_s"] = run_stage("latentsync", [
            LS_PY, "-m", "scripts.inference",
            "--unet_config_path", "configs/unet/stage2.yaml",
            "--inference_ckpt_path", "checkpoints/latentsync_unet.pt",
            "--inference_steps", steps,
            "--guidance_scale", guidance,
            "--enable_deepcache",
            "--video_path", ls_input,
            "--audio_path", audio,
            "--video_out_path", hybrid_mp4], SRC_LS, work)
    timings["vram_peak_mb"] = vram.peak_mb

    shutil.copy2(ditto_mp4, out_dir / "ditto_raw.mp4")
    shutil.copy2(hybrid_mp4, out_dir / "hybrid_256.mp4")
    (out_dir / "timings.json").write_text(json.dumps(timings, indent=2))
    (out_dir / "error.log").unlink(missing_ok=True)  # retry thanh cong -> don log loi cu
    print(f"[{job}] XONG — ditto {timings['ditto_s']}s + latentsync {timings['latentsync_s']}s"
          f" | VRAM peak ~{vram.peak_mb}MB")


jobs = [d for d in sorted(IN_DIR.iterdir()) if d.is_dir()]
if not jobs:
    print(f"Chua co job nao trong {IN_DIR} — tao in/<ten-job>/ gom image.png + audio.wav roi chay lai cell nay.")
for job_dir in jobs:
    try:
        run_job(job_dir)
    except Exception as exc:  # 1 job hong (render, params.json, Drive I/O...) khong duoc giet ca batch
        # run_stage da in tail log loi ra cell; log day du: /content/work/<job>/<stage>.log
        print(f"[{job_dir.name}] LOI {type(exc).__name__}: {exc} — chay tiep job sau")
        try:
            err_dir = OUT_DIR / job_dir.name
            err_dir.mkdir(parents=True, exist_ok=True)
            (err_dir / "error.log").write_text(
                f"{type(exc).__name__}: {exc}\n"
                "Log day du: /content/work/<job>/ditto.log | latentsync.log (xem tail o output cell Buoc 5).\n")
        except OSError as log_exc:
            print(f"[{job_dir.name}] (khong ghi duoc error.log: {log_exc})")


In [ ]:
# Buoc 6: tong ket so do moi job da xong + viec tiep theo tren may local
import json

done = sorted(OUT_DIR.glob("*/timings.json"))
if not done:
    print("Chua co job nao hoan thanh.")
for tj in done:
    t = json.loads(tj.read_text())
    print(f"{tj.parent.name}: ditto {t.get('ditto_s', '?')}s"
          f" + latentsync-256 {t.get('latentsync_s', '?')}s"
          f" | VRAM peak {t.get('vram_peak_mb', '?')}MB | {t.get('gpu', '')}")
print()
print("Buoc tiep theo (tren may local):")
print(r"  1. Tai out/<job>/hybrid_256.mp4 ve outputs\colab")
print(r"  2. .venv\Scripts\python.exe scripts\matte_video.py --video outputs\colab\hybrid_256.mp4")
print(r"  3. .venv\Scripts\python.exe scripts\sync_metrics.py --video outputs\lipsync_green_<runid>.mp4")


## Sự cố thường gặp

| Triệu chứng | Nguyên nhân / cách xử |
|---|---|
| Cell Bước 0 assert fail | Chưa bật GPU runtime — Runtime → Change runtime type → T4 GPU |
| `CUDA out of memory` ở LatentSync | Job trước chưa nhả VRAM → Runtime → Restart session, chạy lại (checkpoints/venv còn nguyên trong phiên) |
| Cài insightface lâu (vài phút, nhiều log gcc) | Bình thường — build từ source, chỉ lần đầu mỗi phiên |
| Cài venv fail giữa chừng (mạng chập chờn) | Chạy lại cell — sentinel `.install-complete` chỉ được ghi khi cài xong, nên lần sau cài lại từ đầu chứ không dùng venv dở |
| Lỗi `mask_renderer` / numpy ABI khi import insightface | Hiếm trên Linux; nếu dính: mở `venvs/latentsync/lib/python3.11/site-packages/insightface/app/__init__.py`, bọc `try/except ImportError` quanh import mask_renderer (đã làm y hệt trên máy local Windows) |
| Session chết giữa chừng (12h / idle 90ph) | Chạy lại Run all — setup cells idempotent, job đã xong bị bỏ qua, job dở chạy lại từ đầu |
| GPU khác T4 (L4/A100 khi có Colab Pro) | Chạy bình thường và nhanh hơn; muốn thử config 512 thì đổi 2 chỗ: cell Bước 5 `stage2.yaml` → `stage2_512.yaml` và cell Bước 4 tải ckpt từ `ByteDance/LatentSync-1.6` |

Số đo tham chiếu local (RTX 3060 12GB, clip 20.4s): Ditto ~157s · LatentSync-**512** 157 **phút** (VRAM tràn — lý do notebook này tồn tại). Kỳ vọng T4 với config 256: tổng ≤ ~30 phút/20s.
